In [1]:
import numpy as np 
import soundfile as sf
import librosa
import pandas as pd 
from pathlib import Path
import pickle 
import IPython.display as ipd
import sys 
sys.path.append('../../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 
import soxr

# Make stimuli for 2023 mono word recognition experiment

## Wanted Conditions:
### SNRs:
-9, -6, -3, 0, 3, inf 
### Distractor conditions:
- 1-talker
- 4-talker 
- Speech-shaped noise
- Modulated Masker 
- Music
- Natural scenes  
- 8-talker babble 

Will be using foregrounds and cues from manifest:
`/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_experiment_v00/cue_and_target_manifest.pdpkl`

Stim will be moved to `/om/user/imgriff/datasets/human_word_rec_SWC_2023`

In [2]:
# manifest = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_experiment_v00/cue_and_target_manifest.pdpkl')

In [3]:
# talker_gender_dict = manifest_all_words.groupby('client_id').gender.unique().to_dict()
# talker_gender_dict = {talker:gender[0] for talker,gender in talker_gender_dict.items()}

In [4]:
stim_out_path = Path('/om/user/imgriff/datasets/human_word_rec_SWC_2023')
manifest = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/full_eval_trial_manifest_new_fnames.pdpkl')
manifest.shape

(1440, 35)

In [5]:
# drop columns with distractor in name
unique_manifest = manifest[[col for col in manifest.columns if 'distractor' not in col]]
# drop "gender_cond_td" column from manifest
unique_manifest = unique_manifest.drop(columns=['gender_cond_td'])

# drop duplicate rows from manifest
unique_manifest = unique_manifest.drop_duplicates().reset_index(drop=True)
print(unique_manifest.shape) # should be 720 x 21


(720, 21)


In [6]:
# save manifest 
# unique_manifest.to_pickle("/om/user/imgriff/datasets/human_word_rec_SWC_2023/unique_cue_and_target_manifest_for_human_expmnt.pdpkl")

In [7]:
unique_manifest = pd.read_pickle("/om/user/imgriff/datasets/human_word_rec_SWC_2023/unique_cue_and_target_manifest_for_human_expmnt.pdpkl")

#### Get distractor manifest for 1 and 4 distractors conditions


In [8]:
fn_pkl_dst = '/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl'
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))
words = list(word_dict.keys())
words = [word.replace("'", "") for word in words]
manifest_all_words = pd.read_pickle(fn_pkl_dst)
# filter out words not in 'words' list
manifest_all_words = manifest_all_words[manifest_all_words['word'].isin(words)]
word_counts = manifest_all_words['word'].value_counts()

# distractor manifest - talkers & words not in test foreground set but still in vocab
target_vocab = unique_manifest['word'].unique()
distractor_manifest = manifest_all_words[~manifest_all_words['word'].isin(target_vocab)]
# filter out target talkers
distractor_manifest = distractor_manifest[~distractor_manifest.client_id.isin(unique_manifest.client_id.unique())]

In [9]:
# # save as words as json 
# import json 
# fn_dst = '/om/user/imgriff/datasets/human_word_rec_SWC_2023/dictionary_cv_800_words.json'
# with open(fn_dst, 'w') as f:
#     json.dump({"dictionary":words}, f)


In [10]:
distractor_manifest.shape

(22262, 13)

In [11]:
# Make condition dict 
import itertools 
import pickle 
import numpy as np 
snrs = np.arange(-9, 4, 3).tolist() # -9, -6, -3, 0, 3

conditions = ['background_musdb18hq',
              'background_cv08talkerbabble',
              'background_issnstationary',
              'background_issnfestenplomp',
              'background_audioset',
              'background_ieeeaaspcasa',
              "1-talker",
              "4-talker"]

condition_pairs = list(itertools.product(conditions, snrs))
condition_pairs.append(('SILENCE', 'inf'))
cond_ix_dict = {ix:cond for ix, cond in enumerate(condition_pairs)}

out_name = "human_saddler_attn_expmt_cond_map.pkl"
# write condition dict as pickle 
with open(out_name, 'wb') as f:
    pickle.dump(cond_ix_dict, f)


In [34]:
cond_ix_dict

{0: ('background_musdb18hq', -9),
 1: ('background_musdb18hq', -6),
 2: ('background_musdb18hq', -3),
 3: ('background_musdb18hq', 0),
 4: ('background_musdb18hq', 3),
 5: ('background_cv08talkerbabble', -9),
 6: ('background_cv08talkerbabble', -6),
 7: ('background_cv08talkerbabble', -3),
 8: ('background_cv08talkerbabble', 0),
 9: ('background_cv08talkerbabble', 3),
 10: ('background_issnstationary', -9),
 11: ('background_issnstationary', -6),
 12: ('background_issnstationary', -3),
 13: ('background_issnstationary', 0),
 14: ('background_issnstationary', 3),
 15: ('background_issnfestenplomp', -9),
 16: ('background_issnfestenplomp', -6),
 17: ('background_issnfestenplomp', -3),
 18: ('background_issnfestenplomp', 0),
 19: ('background_issnfestenplomp', 3),
 20: ('background_audioset', -9),
 21: ('background_audioset', -6),
 22: ('background_audioset', -3),
 23: ('background_audioset', 0),
 24: ('background_audioset', 3),
 25: ('background_ieeeaaspcasa', -9),
 26: ('background_ieee

In [12]:
path_to_bg_stim = Path("/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/")

In [13]:
manifest.head()

,distractor_client_id,distractor_clip_dur_in_s,distractor_clip_end_in_s,distractor_clip_start_in_s,distractor_corpus,distractor_gender,distractor_gender_int,distractor_split,distractor_split_int,distractor_sr,...,total_file_duration_in_s,word,cue_word,cue_src_ix,cue_client_id,cue_src_fn,cue_clip_start_in_s,cue_clip_end_in_s,gender_cond_td,cue_clip_dur_in_s
0,popularoutcast,0.44,359.92,359.48,swc,female,0,NaN,0,44100,...,2207.180045,these,dramatic,992746,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1147.51,1147.96,female_female,0.45
1,matthewdgonzalez,0.49,584.65,584.16,swc,male,1,NaN,0,44100,...,2207.180045,these,dramatic,992746,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1147.51,1147.96,female_male,0.45
2,flyingtoaster,0.17,1711.42,1711.25,swc,female,0,NaN,0,44100,...,2207.180045,simply,allows,992957,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1905.81,1906.30,female_female,0.49
3,warmvoiceover,0.76,487.07,486.31,swc,male,1,NaN,0,44100,...,2207.180045,simply,allows,992957,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1905.81,1906.30,female_male,0.49
4,popularoutcast,0.56,432.21,431.65,swc,female,0,NaN,0,44100,...,3793.117460,death,with,999423,99of9-toby-hudson,/om/user/imgriff/datasets/spatial_audio_pipeli...,642.92,643.09,male_female,0.17


In [14]:
unique_manifest['word_int'] = unique_manifest['word'].map(word_dict)

In [15]:
# Crop waveforms to given duration keeping center segment 
def crop_wav(wav, sr, duration):
    n_samples = int(duration*sr)
    start_idx = int((len(wav) - n_samples)/2)
    end_idx = int(start_idx + n_samples)
    return wav[start_idx:end_idx]

def get_excerpt(dfi, dur=3.0, sr=50000, pad_with_context=True, jitter_fraction=0):
    """
    This function loads an audio file and excerpts a clip with the specified
    duration. Target durations that exceed clip boundaries are handled with
    zero-padding (applied to all signals but sliced away when not needed).
    This function also handles resampling (via soxr) and re-scaling.
    """
    jitter_in_s = 0
    jitter_via_zero_padding = True
    if dfi.clip_dur_in_s > dur:
        # Take a random segment if clip duration is longer than excerpt
        clip_start_in_s = np.random.uniform(
            low=dfi.clip_start_in_s,
            high=dfi.clip_start_in_s + dfi.clip_dur_in_s - dur,
            size=None)
        clip_end_in_s = clip_start_in_s + dur
        jitter_via_zero_padding = False
    else:
        # Temporally jitter clip by extending either start or end time
        jitter_in_s = np.random.uniform(
            low=-dfi.clip_dur_in_s * jitter_fraction,
            high=dfi.clip_dur_in_s * jitter_fraction,
            size=None)
        if pad_with_context:
            # If using context, adjust clip start and end times to account for jitter and context
            if jitter_in_s > 0:
                clip_start_in_s = dfi.clip_start_in_s - (2 * np.abs(jitter_in_s))
                clip_end_in_s = dfi.clip_end_in_s
            else:
                clip_start_in_s = dfi.clip_start_in_s
                clip_end_in_s = dfi.clip_end_in_s + (2 * np.abs(jitter_in_s))
            clip_dur_in_s = clip_end_in_s - clip_start_in_s
            jitter_via_zero_padding = False
            context_pad_in_s = (dur - clip_dur_in_s) / 2
        else:
            clip_start_in_s = dfi.clip_start_in_s
            clip_end_in_s = dfi.clip_end_in_s
            context_pad_in_s = 0
        clip_start_in_s = clip_start_in_s - context_pad_in_s
        clip_end_in_s = clip_end_in_s + context_pad_in_s
    clip_dur_in_s = clip_end_in_s - clip_start_in_s
    # Load audio, pad, slice with indexes that account for padding
    load_full_file = True
    if (clip_start_in_s >= 0) and (clip_end_in_s < dfi.total_file_duration_in_s):
        # Attempt to read only the specified excerpt
        myfile = sf.SoundFile(dfi.src_fn)
        if myfile.seekable():
            src_sr = myfile.samplerate
            frame_start = int(np.round(clip_start_in_s * src_sr))
            frames = int(np.round(clip_dur_in_s * src_sr))
            myfile.seek(frame_start)
            y = myfile.read(frames, always_2d=True)
            y = np.mean(y, axis=1)
            load_full_file = False
    if load_full_file:
        # If impossible, read full audio file
        y, src_sr = sf.read(dfi.src_fn, always_2d=True)
        y = np.mean(y, axis=1)
        frame_start = int(np.round(clip_start_in_s * src_sr))
        frames = int(np.round(clip_dur_in_s * src_sr))
        if frame_start < 0:
            y = np.pad(y, [-frame_start, 0])
            frame_start = 0
        if frame_start + frames > len(y):
            y = np.pad(y, [0, frame_start + frames - len(y)])
        y = y[frame_start : frame_start + frames]
    # Resample from src_sr to sr
    y = soxr.resample(y, src_sr, sr).astype(np.float32)
    # If not yet jittered, apply jitter at end via asymmetric zero-padding
    if jitter_via_zero_padding:
        jitter_pad_width = int(np.round(2 * np.abs(jitter_in_s) * sr))
        if jitter_in_s > 0:
            y = np.pad(y, [jitter_pad_width, 0]).astype(np.float32)
        else:
            y = np.pad(y, [0, jitter_pad_width]).astype(np.float32)
    # Zero-pad or trim to length (fixes off by one errors)
    y = util_audio.pad_or_trim_to_len(y, int(dur * sr))
    y = np.nan_to_num(y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return y

def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factor

def rms_normalize_db(wav, dBSPL, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    new_rms = 20e-6 * np.power(10, dBSPL/20)
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor



In [16]:
pd.DataFrame.from_records

<bound method DataFrame.from_records of <class 'pandas.core.frame.DataFrame'>>

In [17]:
female_samps = unique_manifest[unique_manifest.gender == 'female'].sample(180)
female_words = female_samps.word.unique()
male_samps = unique_manifest[(unique_manifest.gender == "male") & ~unique_manifest.word.isin(female_words)].sample(180)
stim_df = pd.concat([female_samps, male_samps], axis=0, ignore_index=True)

In [18]:
stim_df.gender.value_counts(), stim_df.word.nunique()

(female    180
 male      180
 Name: gender, dtype: int64,
 360)

In [19]:
set(male_samps.word.unique()).intersection(set(female_words))

set()

In [20]:
unique_manifest.sample(1).src_fn.values[0]

'/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/target_excerpts/added_mangst.wav'

In [30]:
taret_sr = 44100

target_wav, _ = librosa.load(unique_manifest.sample(1).src_fn.values[0], sr=taret_sr)
bg_signal = util_audio.spectrally_matched_noise(target_wav, taret_sr)
# if "festenplomp" in condition:
donor_wav = distractor_manifest.sample(1).apply(lambda x: get_excerpt(x, dur=2, sr=44100, pad_with_context=True, jitter_fraction=0), axis=1).values[0]


In [31]:
donor_wav

array([0.00860596, 0.00752258, 0.0059967 , ..., 0.01808167, 0.01797485,
       0.01803589], dtype=float32)

In [32]:
bg_signal = util_audio.festen_plomp_fluctuating_noise(donor_wav, bg_signal, sr=44100)


In [33]:
bg_signal.shape

(88200,)

In [93]:
from tqdm.auto import tqdm
SR = 44100
# timing in seconds 
trial_dur = 4.5
signal_dur = 2
isi = 0.5
win_dur = 10 # ms 
mixture_onset = int((isi + signal_dur) * SR) # in frames
# make dir 
stim_out_path = Path('/om/user/imgriff/datasets/human_word_rec_SWC_2023/sounds')

stim_out_path.mkdir(parents=True, exist_ok=True)
records = []
for cond_ix, (condition, snr) in tqdm(cond_ix_dict.items()):
    bg_stim = list((path_to_bg_stim / condition).glob('*.wav'))
    female_samps = unique_manifest[unique_manifest.gender == 'female'].sample(180)
    female_words = female_samps.word.unique()
    male_samps = unique_manifest[(unique_manifest.gender == "male") & ~unique_manifest.word.isin(female_words)].sample(180)
    wanted_manifest = pd.concat([female_samps, male_samps], axis=0, ignore_index=True)

    cond_records = []
    for row in tqdm(wanted_manifest.itertuples()):
        record = {}
        trial_signal = np.zeros((int(SR * trial_dur)),dtype=np.float32)
        ix = row.Index # ix for background in 
        # are already cut/resampled/windowed
        cue_wav, cue_sr = librosa.load(row.cue_src_fn, sr=SR)
        target_wav, target_sr = librosa.load(row.src_fn, sr=SR)
        record['target_sr'] = target_sr
        record['cue_sr'] = cue_sr
        record['target_fn'] = row.src_fn
        record['cue_fn'] = row.cue_src_fn
        record['word'] = row.word
        record['word_int'] = row.word_int
        record['condition'] = condition
        record['snr'] = snr
        record['src_ix'] = row.src_ix
        record['client_id'] = row.client_id
        record['target_gender'] = row.gender

        # get talker f0 - bound to range of human voice 
        target_f0 = librosa.pyin(target_wav, fmin=80, fmax=300, sr=SR, fill_na=np.nan)
        avg_target_f0 = np.nanmean(target_f0)
        record['target_f0'] = avg_target_f0
        if "1-talker" in condition:
            distractor_gender = 'male' if ix < 180 else 'female'
            # get existing distractor signal used in binaural experiment
            distractor_eg = manifest[(manifest["src_ix"] == unique_manifest.src_ix[0]) & (manifest['distractor_gender'] == distractor_gender)].iloc[0]
            # is already cut/resampled/windowed
            bg_signal, _ = librosa.load(distractor_eg.distractor_src_fn, sr=SR)
            bg_talker_f0 = librosa.pyin(bg_signal, fmin=80, fmax=300, sr=SR, fill_na=np.nan)
            avg_bg_f0 = np.nanmean(bg_talker_f0)
            record['distractor_fn'] = distractor_eg.distractor_src_fn
            record['distractor_f0'] = avg_bg_f0
            record['distractor_gender'] = distractor_gender
         
        elif "4-talker" in condition:
            bg_examples = distractor_manifest.sample(4).apply(lambda x: get_excerpt(x, dur=2, sr=44100, pad_with_context=True, jitter_fraction=0), axis=1)
            bg_talkers = util_audio.set_dBSPL(np.stack(bg_examples.values), 60) # set to 60 dB SPL
            bg_signal = bg_talkers.sum(axis=0) # set mixture to 60 dB
            record['distractor_fn'] = list(bg_examples.src_fn.values)
            record['distractor_f0'] = [np.nanmean(librosa.pyin(bg_talker, fmin=80, fmax=300, sr=SR, fill_na=np.nan)) for bg_talker in bg_talkers]
            record['distractor_gender'] = list(bg_examples.gender.values)
        elif "issn" in condition:
            bg_signal = util_audio.spectrally_matched_noise(target_wav, target_sr)
            if "festenplomp" in condition:
                donor_wav = distractor_manifest.sample(1).apply(lambda x: get_excerpt(x, dur=2,
                                                                                      sr=44100,
                                                                                      pad_with_context=True,
                                                                                      jitter_fraction=0), axis=1).values[0]
                bg_signal = util_audio.festen_plomp_fluctuating_noise(donor_wav, bg_signal, sr=SR)
        else:
            ix  = ix % 360
            bg_signal, _ = librosa.load(bg_stim[ix], sr=SR)
            # crop and window to signal duration
            bg_signal = util_audio.ramp_hann(util_audio.pad_or_trim_to_len(bg_signal, int(2*SR)), win_dur, samplerate=SR)
        
        mixture = util_audio.combine_signal_and_noise(target_wav, bg_signal, snr)
        mixture = util_audio.ramp_hann(mixture, win_dur, samplerate=SR)

        mixture, mixture_scale_factor = rms_normalize_db(mixture, 60)
        # set cue to same level as target post scaling 
        cue_wav = util_audio.set_dBSPL(cue_wav, 60)
        cue_wav *= mixture_scale_factor

        # add cue to trial signal
        trial_signal[:len(cue_wav)] += cue_wav    
        trial_signal[mixture_onset:] += mixture
        trial_signal = util_audio.set_dBSPL(trial_signal, 60)
        # save trial signal
        # f string with digint padded to 3 places
        out_name = stim_out_path / f"condition_{cond_ix}"/ f'{row.word_int:03}.wav'
        # make out name directory if it doesn't exist
        out_name.parent.mkdir(parents=True, exist_ok=True)
        sf.write(out_name, trial_signal, samplerate=SR)
        record['mixture_fn'] = out_name
        cond_records.append(record)

    # break


  0%|          | 0/41 [00:00<?, ?it/s]

0it [00:00, ?it/s]

KeyboardInterrupt: 

In [54]:
ipd.Audio(trial_signal, rate=SR, normalize=True)

## Remap catch trials 

NameError: name 'old_to_new_word_map' is not defined

In [66]:
old_word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_word_int_label_dict.pkl", 'rb'))
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))

ix_to_word_map = {ix:word for word,ix in old_word_dict.items()}

# remap catch trial names from old  to new 
old_stim_trials = [
    250,
    55,
    152,
    214,
    555,
    43,
    142,
    39,
    495,
    204,
    149,
    423,
    80,
    140,
    522,
    122,
    60,
    155,
    40,
    295,
    117,
    309,
    71,
    73,
    608,
    321,
    254,
    44,
    93,
    714,
    138,
    790,
    186,
    150,
    145,
    56,
    288,
    285,
    366,
    562,
    391,
    193,
    95,
    86,
    789,
    464,
    106,
    301
]

new_map = {ix_to_word_map[ix]: word_dict[ix_to_word_map[ix]] for ix in old_stim_trials}

cath_trial_words = [ix_to_word_map[ix] for ix in old_stim_trials]
# cath_trial_words
# new_map

In [69]:
list(new_map.values())

[15,
 703,
 155,
 292,
 706,
 14,
 452,
 479,
 635,
 501,
 187,
 68,
 794,
 768,
 558,
 60,
 713,
 64,
 253,
 513,
 659,
 710,
 770,
 788,
 415,
 586,
 229,
 697,
 714,
 705,
 787,
 197,
 42,
 33,
 771,
 700,
 58,
 617,
 707,
 162,
 289,
 379,
 0,
 90,
 9,
 712,
 492,
 27]

In [35]:
from IPython.display import Audio
from pathlib import Path
catch_trial_stim_path = list(Path("/mindhive/mcdermott/www/imgriff/msjspsych/cocktail_party_word_recognition/stim/catch_trial_stim/").glob("*wav"))

In [73]:
path_to_bg_stim = list(Path("/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/background_issnfestenplomp/").glob("*.wav"))

In [75]:
Audio(path_to_bg_stim[0])

In [54]:
new_f_names = []
for path in catch_trial_stim_path:
    stem_ = path.stem
    word_int = stem_.split('_')[0]
    new_int = word_dict[ix_to_word_map[int(word_int)]]
    new_stem = stem_.replace(word_int, str(new_int))+".wav"
    new_f_names.append(path.parent / new_stem)


In [55]:
# rename files in catch_trial_stim_path with names in new_f_names
for old, new in zip(catch_trial_stim_path, new_f_names):
    old.rename(new)

In [25]:
print(ix_to_word_map[int(catch_trial_stim_path[0].stem.split("_")[0])])
Audio(catch_trial_stim_path[0])

place


In [40]:
stim = "/om/user/imgriff/datasets/human_word_rec_SWC_2023/sounds/condition_19/040.wav"
Audio(stim)

In [41]:
stim = "/mindhive/mcdermott/www/imgriff/msjspsych/cocktail_party_word_recognition/stim/sounds/condition_19/040.wav"
Audio(stim)

In [29]:
distractor_manifest

,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,sr,src_fn,total_file_duration_in_s,word
105958,frogbrains,0.39,2.89,2.50,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,215.500045,video
105975,frogbrains,0.33,17.74,17.41,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,215.500045,first
105986,frogbrains,0.34,37.55,37.21,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,215.500045,player
105996,frogbrains,0.30,44.10,43.80,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,215.500045,player
106009,frogbrains,0.30,51.49,51.19,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,215.500045,later
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1006249,supa6661,0.49,422.98,422.49,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,588.318667,appears
1006251,supa6661,0.73,458.22,457.49,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,588.318667,exist
1006259,supa6661,0.31,464.55,464.24,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,588.318667,usually
1006271,supa6661,0.40,472.82,472.42,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,588.318667,places
